<a href="https://colab.research.google.com/github/kowsik005/Unsupervised_ML_project/blob/main/Association_rule.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# ============================================================
# K-MEANS CLUSTERING - UNSUPERVISED LEARNING
# Google Colab - NO INSTALLATION REQUIRED
# ============================================================

# 1. Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from mlxtend.preprocessing import TransactionEncoder # Added for one-hot encoding

# ============================================================
# 2. Load Dataset
# ============================================================

# Change this to your CSV file name
file_path = "/content/Market_Basket_Optimisation-selected-columns.csv"

# Load the dataset without a header, assuming each row is a transaction
df_raw = pd.read_csv(file_path, header=None) # Changed: Load with header=None

print("Raw Dataset Shape:", df_raw.shape)
print("\nFirst 5 Rows of Raw Data:")
display(df_raw.head())

# ============================================================
# 3. Convert Dataset into Transactions (list of lists)
# ============================================================

transactions = []
for row_idx in range(df_raw.shape[0]):
    # Filter out NaN values and convert to string for each item in the row
    items = [str(item).strip() for item in df_raw.iloc[row_idx].dropna() if str(item).strip() != '']
    if items: # Only add non-empty transactions
        transactions.append(items)

print("\nNumber of transactions:", len(transactions))
print("Example transaction:", transactions[0])

# ============================================================
# 4. One-Hot Encode Transactions
# ============================================================

te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)
df = pd.DataFrame(te_array, columns=te.columns_) # df now stores the one-hot encoded data

print("\nOne-Hot Encoded Dataset Shape:", df.shape)
print("\nFirst 5 Rows of One-Hot Encoded Data:")
display(df.head())

# ============================================================
# 5. Check Dataset (now numerical)
# ============================================================

print("\nColumn Names (Items):")
print(df.columns.tolist())

print("\nMissing Values (should be none after one-hot encoding):")
print(df.isnull().sum().sum()) # Should print 0

# ============================================================
# 6. Prepare data for clustering
# ============================================================

X = df.copy() # X is now the one-hot encoded DataFrame, which is numerical

print("\nFeatures (X) shape:", X.shape)

# ============================================================
# 7. Standardize Data
# ============================================================

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

print("\nData successfully standardized. Shape:", X_scaled.shape)

# ============================================================
# 8. ELBOW METHOD
# ============================================================

inertia = []

# If there's only one transaction, K-Means is not meaningful for K>1
# Adjust range if number of transactions is very small
if len(transactions) < 2:
    print("\nWarning: Not enough unique transactions for meaningful K-Means clustering (need at least 2). Skipping Elbow Method.")
    K_range = [1] # Cannot calculate inertia meaningfully for K>1
else:
    K_range = range(2, min(11, len(transactions) + 1))

if K_range and K_range[0] != 1: # Only proceed if K is not just 1
    for k in K_range:

        model = KMeans(
            n_clusters=k,
            random_state=42,
            n_init=10
        )

        model.fit(X_scaled)

        inertia.append(model.inertia_)

    # Plot Elbow Curve
    plt.figure(figsize=(8, 5))
    plt.plot(
        K_range,
        inertia,
        marker="o"
    )
    plt.xlabel("Number of Clusters (K)")
    plt.ylabel("Inertia")
    plt.title("Elbow Method")
    plt.grid(True)
    plt.show()
else:
    print("\nNot enough data points to perform Elbow Method for K > 1.")


# ============================================================
# 9. SILHOUETTE SCORE
# ============================================================

silhouette_scores = []

if len(transactions) < 2:
    print("\nWarning: Not enough unique transactions for meaningful K-Means clustering (need at least 2). Skipping Silhouette Score.")
    best_k = 1 # Default to 1 if no clustering can be done
elif len(transactions) >=2:
    # Recalculate K_range to ensure it starts from 2 for silhouette score
    K_range_silhouette = range(2, min(11, len(transactions) + 1))
    if not K_range_silhouette:
        print("\nNot enough data points to calculate Silhouette Score for K >= 2.")
        best_k = 1
    else:
        for k in K_range_silhouette:
            model = KMeans(
                n_clusters=k,
                random_state=42,
                n_init=10
            )

            labels = model.fit_predict(X_scaled)

            score = silhouette_score(
                X_scaled,
                labels
            )

            silhouette_scores.append(score)

        # Display scores
        print("\nSilhouette Scores:")
        for k, score in zip(K_range_silhouette, silhouette_scores):
            print(
                f"K = {k} --> Silhouette Score = {score:.4f}"
            )

        # Find best K based on silhouette score
        best_k = K_range_silhouette[np.argmax(silhouette_scores)]
        print("\nBest Number of Clusters (from Silhouette Score):", best_k)

        # Plot Silhouette Scores
        plt.figure(figsize=(8, 5))
        plt.plot(
            K_range_silhouette,
            silhouette_scores,
            marker="o"
        )
        plt.xlabel("Number of Clusters (K)")
        plt.ylabel("Silhouette Score")
        plt.title("Silhouette Score")
        plt.grid(True)
        plt.show()
else:
    print("\nNot enough data points to calculate Silhouette Score.")
    best_k = 1

# ============================================================
# 10. Train Final K-Means Model
# ============================================================

# Ensure best_k is at least 1, but if data is too sparse, warn the user.
if best_k > 1 and len(transactions) >= 2:
    kmeans = KMeans(
        n_clusters=best_k,
        random_state=42,
        n_init=10
    )
    clusters = kmeans.fit_predict(X_scaled)
    print("\nFinal K-Means model trained with best_k =", best_k)
else:
    clusters = np.zeros(X_scaled.shape[0], dtype=int) # Assign all to cluster 0 if only one transaction
    best_k = 1
    print("\nSkipping K-Means model training due to insufficient data for clustering (best_k=1 or less than 2 transactions).")
    print("All data points assigned to a single cluster (Cluster 0).")

# ============================================================
# 11. Add Cluster to Original Dataset
# ============================================================

df["Cluster"] = clusters

print("\nClustered Dataset (first 20 rows):")
display(df.head(20))

# ============================================================
# 12. Cluster Counts
# ============================================================

print("\nNumber of Data Points in Each Cluster:")
print(df["Cluster"].value_counts().sort_index())

# ============================================================
# 13. Cluster Centers
# ============================================================

if best_k > 1:
    centers = scaler.inverse_transform(
        kmeans.cluster_centers_
    )

    centers_df = pd.DataFrame(
        centers,
        columns=X.columns # Use original one-hot encoded column names
    )

    centers_df.index.name = "Cluster"

    print("\nCluster Centers (mean presence of each item in cluster):")
    display(centers_df)
else:
    print("\nCluster centers not calculated for best_k <= 1.")

# ============================================================
# 14. PCA FOR 2D VISUALIZATION
# ============================================================

if best_k > 1 and X_scaled.shape[0] > 1:
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_scaled)
    print("\nPCA performed for visualization.")
else:
    print("\nSkipping PCA for visualization due to insufficient data or best_k <= 1.")
    X_pca = np.array([])

# ============================================================
# 15. Visualize Clusters
# ============================================================

if best_k > 1 and X_pca.shape[0] > 0:
    plt.figure(figsize=(10, 7))

    for cluster in range(best_k):
        points = X_pca[clusters == cluster]
        plt.scatter(
            points[:, 0],
            points[:, 1],
            label=f"Cluster {cluster}",
            s=50
        )

    plt.xlabel("Principal Component 1")
    plt.ylabel("Principal Component 2")
    plt.title("K-Means Clustering")
    plt.legend()
    plt.grid(True)
    plt.show()
else:
    print("\nSkipping cluster visualization due to insufficient data or best_k <= 1.")

# ============================================================
# 16. Final Silhouette Score
# ============================================================

if best_k > 1 and X_scaled.shape[0] >= 2:
    final_score = silhouette_score(
        X_scaled,
        clusters
    )
    print(
        f"\nFinal Silhouette Score: {final_score:.4f}"
    )
else:
    print("\nFinal Silhouette Score not calculated for best_k <= 1 or less than 2 data points.")

# ============================================================
# 17. Cluster Summary
# ============================================================

print("\n========== CLUSTER SUMMARY ==========")

if best_k > 1:
    summary = df.groupby("Cluster")[X.columns].mean()
    display(summary)
else:
    print("\nCluster summary not generated for best_k <= 1. All data points are in a single cluster.")
    # Display mean of all items if all in one cluster
    print("Mean presence of each item across all data points:")
    display(X.mean().to_frame(name="Average Item Presence"))

# ============================================================
# 18. Save Result
# ============================================================

output_path = "/content/clustered_dataset.csv"

df.to_csv(
    output_path,
    index=False
)

print("\nClustered dataset saved successfully:")
print(output_path)


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

Raw Dataset Shape: (125, 10)

First 5 Rows of Raw Data:


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,0,1,2,3,4,5,6,7,8,9
0,shrimp,almonds,avocado,vegetables mix,green grapes,whole weat flour,yams,cottage cheese,energy drink,tomato juice
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

...

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

,Average Item Presence
almonds,1.0
avocado,1.0
cottage cheese,1.0
energy drink,1.0
green grapes,1.0
shrimp,1.0
tomato juice,1.0
vegetables mix,1.0
whole weat flour,1.0
yams,1.0


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag


Clustered dataset saved successfully:
/content/clustered_dataset.csv


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

It appears the CSV file is not comma-separated, but rather contains items in separate columns, with each row representing a transaction and each item taking up a distinct column. Pandas' `read_csv` by default expects comma-separated values. Given this structure, we should load the data without a specific delimiter and treat each column as an item in the transaction. This will ensure all columns are loaded correctly.